In [116]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
import random

In [117]:
class TranslationDataset(Dataset):
    def __init__(self, pairs):
        self.pairs = pairs
        self.input_vocab = {'<PAD>': 0, '<BOS>': 1, '<EOS>': 2}
        self.output_vocab = {'<PAD>': 0, '<BOS>': 1, '<EOS>': 2}

        for eng, rus in pairs:
            for word in eng.split():
                if word not in self.input_vocab:
                    self.input_vocab[word] = len(self.input_vocab)
            for word in rus.split():
                if word not in self.output_vocab:
                    self.output_vocab[word] = len(self.output_vocab)

        self.input_vocab_inv = {v: k for k, v in self.input_vocab.items()}
        self.output_vocab_inv = {v: k for k, v in self.output_vocab.items()}

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        eng, rus = self.pairs[idx]
        input_seq = [self.input_vocab.get(w, 2) for w in eng.split()]
        output_seq = [self.output_vocab.get(w, 2) for w in rus.split()]
        return torch.tensor(input_seq), torch.tensor(output_seq)

# Функция для сборки batch с паддингом
def collate_fn(batch):
    input_seqs, target_seqs = zip(*batch)
    input_padded = pad_sequence(input_seqs, batch_first=True, padding_value=0)
    target_padded = pad_sequence(target_seqs, batch_first=True, padding_value=0)
    return input_padded, target_padded

In [118]:
def load_data(filename='rus.txt', n_samples=10000):
    df = pd.read_csv(filename, sep='\t', header=None)

    if len(df.columns) == 3:
        df = df[[0, 1]]
        df.columns = ['en', 'ru']
    else:
        df.columns = ['en', 'ru']

    df = df.head(n_samples)
    df['ru'] = '<BOS> ' + df['ru'] + ' <EOS>'

    pairs = list(zip(df['en'].str.lower(), df['ru'].str.lower()))
    return pairs

In [119]:
# класс модели
class Seq2Seq(nn.Module):
    def __init__(self, input_size, output_size, hidden_size=128,
                 num_layers=1, cell_type='gru'):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.cell_type = cell_type

        self.encoder_emb = nn.Embedding(input_size, hidden_size)
        self.decoder_emb = nn.Embedding(output_size, hidden_size)

        if cell_type == 'gru':
            self.encoder_rnn = nn.GRU(hidden_size, hidden_size, num_layers, batch_first=True)
            self.decoder_rnn = nn.GRU(hidden_size, hidden_size, num_layers, batch_first=True)
        else:
            self.encoder_rnn = nn.LSTM(hidden_size, hidden_size, num_layers, batch_first=True)
            self.decoder_rnn = nn.LSTM(hidden_size, hidden_size, num_layers, batch_first=True)

        self.fc = nn.Linear(hidden_size, output_size)

    def encode(self, input_seq):
        enc_emb = self.encoder_emb(input_seq)
        if self.cell_type == 'gru':
            _, enc_hidden = self.encoder_rnn(enc_emb)
        else:
            _, (enc_hidden, _) = self.encoder_rnn(enc_emb)
        return enc_hidden

    def decode_step(self, dec_input, hidden, cell=None):
        dec_emb = self.decoder_emb(dec_input)
        if self.cell_type == 'gru':
            dec_out, hidden = self.decoder_rnn(dec_emb, hidden)
            return dec_out, hidden, None
        else:
            dec_out, (hidden, cell) = self.decoder_rnn(dec_emb, (hidden, cell))
            return dec_out, hidden, cell

    def forward(self, input_seq, target_seq=None, teacher_forcing_ratio=1.0):
        enc_hidden = self.encode(input_seq)
        cell = torch.zeros_like(enc_hidden) if self.cell_type == 'lstm' else None

        if target_seq is not None:
            target_len = target_seq.size(1) - 1
            outputs = []
            dec_input = target_seq[:, 0:1]

            for t in range(target_len):
                dec_out, enc_hidden, cell = self.decode_step(dec_input, enc_hidden, cell)
                out = self.fc(dec_out)
                outputs.append(out)

                if random.random() < teacher_forcing_ratio:
                    dec_input = target_seq[:, t+1:t+2]
                else:
                    dec_input = out.argmax(dim=2)

            return torch.cat(outputs, dim=1)
        else:
            batch_size = input_seq.size(0)
            dec_input = torch.full((batch_size, 1), 1, dtype=torch.long, device=input_seq.device)
            outputs = []

            for _ in range(20):
                dec_out, enc_hidden, cell = self.decode_step(dec_input, enc_hidden, cell)
                out = self.fc(dec_out)
                outputs.append(out)
                dec_input = out.argmax(dim=2)

            return torch.cat(outputs, dim=1)

In [120]:
def train_model(pairs, config, epochs=100):
    device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')

    dataset = TranslationDataset(pairs)
    train_loader = DataLoader(dataset, batch_size=32, shuffle=True, collate_fn=collate_fn)

    model = Seq2Seq(
        input_size=len(dataset.input_vocab),
        output_size=len(dataset.output_vocab),
        hidden_size=256,
        num_layers=config['layers'],
        cell_type=config['cell_type']
    ).to(device)

    optimizer = optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.CrossEntropyLoss(ignore_index=0)

    print(f"Обучение модели {config['name']}")

    for epoch in range(epochs):
        model.train()
        total_loss = 0

        if epoch < epochs * 0.7:
            tf_ratio = 1.0 - epoch / (epochs * 0.7)
        else:
            tf_ratio = 0.0

        for input_seq, target_seq in train_loader:
            input_seq = input_seq.to(device)
            target_seq = target_seq.to(device)

            optimizer.zero_grad()
            output = model(input_seq, target_seq, teacher_forcing_ratio=tf_ratio)

            loss = criterion(output.view(-1, output.size(-1)),
                           target_seq[:, 1:].reshape(-1))
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        if (epoch + 1) % 5 == 0:
            print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss/len(train_loader)}, TF: {tf_ratio}")

    return model, dataset

In [121]:
def translate(model, dataset, sentence, beam_width=5, max_len=20):
    device = next(model.parameters()).device
    model.eval()

    words = sentence.lower().split()
    input_seq = [dataset.input_vocab.get(w, 2) for w in words]
    input_seq = torch.tensor([input_seq]).to(device)

    with torch.no_grad():
        enc_hidden = model.encode(input_seq)
        cell = torch.zeros_like(enc_hidden) if model.cell_type == 'lstm' else None

        beams = [(1.0, [dataset.output_vocab['<BOS>']], enc_hidden, cell)]

        for _ in range(max_len):
            all_candidates = []

            for score, tokens, hidden, cell_state in beams:
                if tokens[-1] == dataset.output_vocab['<EOS>']:
                    all_candidates.append((score, tokens, hidden, cell_state))
                    continue

                dec_input = torch.tensor([[tokens[-1]]]).to(device)
                dec_out, hidden, cell_state = model.decode_step(dec_input, hidden, cell_state)
                out = model.fc(dec_out)

                top_scores, top_tokens = out[0, -1].topk(beam_width)

                for i in range(beam_width):
                    token = top_tokens[i].item()
                    new_score = score + torch.log(top_scores[i]).item()
                    new_tokens = tokens + [token]
                    all_candidates.append((new_score, new_tokens, hidden, cell_state))

            all_candidates.sort(key=lambda x: x[0], reverse=True)
            beams = all_candidates[:beam_width]

            if all(b[1][-1] == dataset.output_vocab['<EOS>'] for b in beams):
                break

        best_tokens = beams[0][1][1:-1]
        output_words = []
        for t in best_tokens:
            if t == dataset.output_vocab['<EOS>']:
                break
            output_words.append(dataset.output_vocab_inv.get(t, ''))

    return ' '.join(output_words)

In [122]:
pairs = load_data('rus.txt', n_samples=10000)

configs = [
    {'name': 'GRU 1 слой', 'layers': 1, 'cell_type': 'gru'},
    {'name': 'GRU 2 слоя', 'layers': 2, 'cell_type': 'gru'},
    {'name': 'LSTM 1 слой', 'layers': 1, 'cell_type': 'lstm'},
    {'name': 'LSTM 2 слоя', 'layers': 2, 'cell_type': 'lstm'},
]

results = {}

for config in configs:
    model, dataset = train_model(pairs, config, epochs=100)

    print(f"\n{config['name']}:")
    test_sentences = ['hello', 'how are you', 'good night']
    for sent in test_sentences:
        translation = translate(model, dataset, sent)
        print(f" {sent} -> {translation}")

    results[config['name']] = model

Обучение модели GRU 1 слой
Epoch 5/100, Loss: 1.066557843273821, TF: 0.9428571428571428
Epoch 10/100, Loss: 0.6415648641296849, TF: 0.8714285714285714
Epoch 15/100, Loss: 0.6264635116909258, TF: 0.8
Epoch 20/100, Loss: 0.6348647060104833, TF: 0.7285714285714286
Epoch 25/100, Loss: 0.6395749437351959, TF: 0.6571428571428571
Epoch 30/100, Loss: 0.6331893401785781, TF: 0.5857142857142856
Epoch 35/100, Loss: 0.6441678863744766, TF: 0.5142857142857142
Epoch 40/100, Loss: 0.6331589473322177, TF: 0.44285714285714284
Epoch 45/100, Loss: 0.6361708561071573, TF: 0.37142857142857144
Epoch 50/100, Loss: 0.632614908983913, TF: 0.30000000000000004
Epoch 55/100, Loss: 0.6265154316212042, TF: 0.22857142857142854
Epoch 60/100, Loss: 0.6138235665738773, TF: 0.15714285714285714
Epoch 65/100, Loss: 0.6031224036369079, TF: 0.08571428571428574
Epoch 70/100, Loss: 0.589921780097218, TF: 0.014285714285714235
Epoch 75/100, Loss: 0.5825316278507915, TF: 0.0
Epoch 80/100, Loss: 0.5753699513479543, TF: 0.0
Epoch 